# DT2 agent: the livestock methane twin over MCP

DT2 serves enteric and manure methane from England's cattle, sheep and goats on a 1 km grid,
as an MCP server, `dt2_server.py`.

* **Head counts**: Defra June Survey, England, 1 June 2020.
* **Where the animals are**: Malek et al. (2024), Zenodo 10.5281/zenodo.13734518, redistributed
  onto **Living England** grassland (Natural England) and calibrated against the NAEI 1 km map.
* **Emission factors**: the UK GHG Inventory factors that the **Farm Carbon Calculator** uses
  (NID 2025, Annex Tables A 5.1.8, A 5.1.9, A 5.1.11), CO2e with the calculator's AR5 GWP of 28.

This notebook plays the orchestrator: it launches the server over stdio, lists its tools and
calls each of them.

In [1]:
import sys
from pathlib import Path

from mcp import Client, StdioServerParameters

SERVER = Path("dt2_server.py").resolve()
params = StdioServerParameters(command=sys.executable, args=[str(SERVER)])


async def ask(tool, **args):
    """Launch DT2, call one tool and return its structured result."""
    async with Client(params) as client:
        result = await client.call_tool(tool, args)
        if result.is_error:
            raise RuntimeError(result.content[0].text)
        return result.structured_content

## Discover the tools

In [2]:
async with Client(params) as client:
    for t in (await client.list_tools()).tools:
        print(f"{t.name:20s} {t.description[:95]}")

list_categories      List DT2's livestock categories with their inventory methane factors, manure systems, and Engla
farm_methane         Methane of one farm's herd from head counts per category, exactly as the Farm Carbon Calculator
area_methane         Livestock and their methane in an area of England: heads, livestock units, enteric and manure C
methane_scenario     What-if for an area: change herd sizes, the grazing share, anaerobic digestion of cattle manure
methane_hotspots     The 1 km cells with the most livestock methane in an area, with their animals, grassland and th
cell_profile         Livestock, methane and grassland of the single 1 km cell at a point.
grassland_map        Area of every Living England habitat class in an area (ha), and the grazing groups DT2 uses (im
validate_naei        Compare DT2's methane map with the NAEI 2020 1 km agricultural methane map over an area: Pearso
calibration_report   The shipped calibration of the livestock allocation against the NAEI m

## Emission factors and the England herd

The inventory's 2023 factors per head, the manure-system mix behind each manure factor, and the methane potential (VS x Bo x 0.67) DT2 back-calculates from them.

In [3]:
cat = await ask("list_categories")
print(f"{'category':38s} {'enteric':>8s} {'manure':>7s} {'MCF %':>6s} {'P kg':>6s}")
for c in cat["categories"]:
    print(f"{c['name']:38s} {c['enteric_kg_ch4_head_yr']:8.2f} {c['manure_kg_ch4_head_yr']:7.2f} "
          f"{c['mcf_weighted_pct']:6.2f} {c['methane_potential_kg_head_yr']:6.1f}")
print()
for g, f in cat["group_factors_kg_head_yr"].items():
    print(f"{g:11s} enteric {f['enteric']:6.1f}  manure {f['manure']:5.2f} kg CH4/head/yr, {f['lsu']:.2f} LSU/head")

category                                enteric  manure  MCF %   P kg
Dairy cows                               123.28   38.03  12.00  316.9
Dairy heifers                             53.72    5.96   6.55   91.0
Dairy replacements >1 year                50.18    5.78   6.55   88.3
Calves <1 year (dairy-calf factor)        42.53    3.67   6.55   56.0
Beef cows                                 75.99   10.72   6.55  163.7
Beef females for slaughter                48.73    5.70   6.55   87.0
Bulls for breeding                        59.13    8.28   6.55  126.4
Heifers for breeding                      48.23    6.25   6.55   95.4
Steers                                    49.91    5.84   6.55   89.2
Ewes                                       6.98    0.18   1.08   16.7
Rams                                       8.25    0.23   1.01   22.8
Lambs                                      3.05    0.07   1.01    6.9
Goats                                      9.00    0.39   1.08   36.1

dairy       enteric

## A farm, the calculator's way

200 dairy cows, 60 calves and 150 followers; then the same herd grazing 60 % of the year.

In [4]:
herd = {"dairy_cows": 200, "calves": 60, "dairy_replacements": 150}
a = await ask("farm_methane", heads=herd)
b = await ask("farm_methane", heads=herd, pasture_pct={"dairy_cows": 60})
print(f"inventory housing mix: {a['total_kg_ch4']/1000:.1f} t CH4 = {a['t_co2e']:.0f} t CO2e (AR5)")
print(f"cows at pasture 60 %:  {b['total_kg_ch4']/1000:.1f} t CH4 = {b['t_co2e']:.0f} t CO2e")

inventory housing mix: 43.4 t CH4 = 1216 t CO2e (AR5)
cows at pasture 60 %:  39.9 t CH4 = 1118 t CO2e


## England

In [5]:
e = await ask("area_methane")
print(f"{e['cells_km2']:,} km2, {e['lsu']/1e6:.2f} M LSU, {e['total_t_ch4']/1e3:.1f} kt CH4 "
      f"({e['enteric_t_ch4']/e['total_t_ch4']:.0%} enteric) = {e['t_co2e_ar5']/1e6:.2f} Mt CO2e (AR5)")
for g, d in e["groups"].items():
    print(f"  {g:11s} {d['heads']/1e6:6.2f} M head  {d['total_t_ch4']/1e3:6.1f} kt CH4")
gl = e["grassland"]
print(f"Living England grazing land: improved {gl['improved_ha']/1e6:.2f} Mha, semi-natural "
      f"{gl['semi_natural_ha']/1e6:.2f} Mha, rough grazing {gl['rough_grazing_ha']/1e6:.2f} Mha")
print(f"stocking {e['stocking_lsu_per_grazing_ha']:.2f} LSU/ha, {e['kg_ch4_per_grazing_ha']:.0f} kg CH4 per grazing ha")

130,524 km2, 5.16 M LSU, 491.7 kt CH4 (86% enteric) = 13.77 Mt CO2e (AR5)
  dairy         1.89 M head   220.3 kt CH4
  beef_other    3.28 M head   193.7 kt CH4
  sheep        15.03 M head    76.8 kt CH4
  goats         0.09 M head     0.8 kt CH4
Living England grazing land: improved 2.00 Mha, semi-natural 3.32 Mha, rough grazing 1.26 Mha
stocking 0.78 LSU/ha, 75 kg CH4 per grazing ha


## An area: 10 km around Crewe (Cheshire dairy country) and the Lake District fells

In [6]:
crewe = {"lat": 53.10, "lon": -2.44, "radius_km": 10}
lakes = {"lat": 54.50, "lon": -3.10, "radius_km": 10}
for name, area in (("Crewe", crewe), ("Lake District", lakes)):
    r = await ask("area_methane", area=area)
    print(f"{name:14s} {r['total_t_ch4']:7.0f} t CH4, {r['stocking_lsu_per_grazing_ha']:.2f} LSU/ha, "
          f"dairy {r['groups']['dairy']['heads']:7.0f}, sheep {r['groups']['sheep']['heads']:7.0f} head")

Crewe             3536 t CH4, 1.55 LSU/ha, dairy   15715, sheep   17731 head


Lake District      701 t CH4, 0.38 LSU/ha, dairy    1441, sheep   74554 head


## What if: a smaller dairy herd and anaerobic digestion

In [7]:
s = await ask("methane_scenario", area=crewe,
              interventions={"herd_change_pct": {"dairy": -20}, "ad_pct_of_housed": 50})
print(f"baseline {s['baseline']['total_t_ch4']:.0f} t CH4 -> scenario {s['scenario']['total_t_ch4']:.0f} t CH4 "
      f"({s['change_pct']:+.1f} %, {s['change_t_co2e_ar5']:+.0f} t CO2e AR5)")
g = await ask("methane_scenario", area=crewe, interventions={"grassland_loss_pct": 10})
print(f"10 % of grassland converted (animals leave with it): {g['change_pct']:+.1f} %")
f = await ask("methane_scenario", area=crewe, interventions={"enteric_reduction_pct": {"dairy": 15}})
print(f"feed additive, 15 % herd-average enteric cut for dairy: {f['change_pct']:+.1f} %")

baseline 3536 t CH4 -> scenario 2988 t CH4 (-15.5 %, -15343 t CO2e AR5)


10 % of grassland converted (animals leave with it): -10.0 %


feed additive, 15 % herd-average enteric cut for dairy: -6.1 %


## Hotspots, one cell, and the grassland map

In [8]:
h = await ask("methane_hotspots", n=5)
for c in h["cells"]:
    print(f"{c['lat']:.3f}N {c['lon']:.3f}E  DT2 {c['total_t_ch4']:5.1f}  NAEI {c['naei_agriculture_t_ch4']:5.1f} t CH4  "
          f"improved grass {c['improved_grassland_ha']:.0f} ha")
c = await ask("cell_profile", lat=53.07, lon=-2.52)
print("cell near Nantwich:", {k: round(v) for k, v in c["heads"].items()}, f"{c['total_t_ch4']:.1f} t CH4")
gm = await ask("grassland_map", area=lakes)
print({k: round(v) for k, v in gm["grazing_groups_ha"].items()})

51.108N -2.580E  DT2  25.9  NAEI  20.9 t CH4  improved grass 94 ha
53.831N -2.860E  DT2  25.8  NAEI  26.8 t CH4  improved grass 104 ha
52.809N -2.172E  DT2  25.8  NAEI  26.3 t CH4  improved grass 92 ha
54.809N -3.052E  DT2  25.8  NAEI  16.4 t CH4  improved grass 109 ha
51.163N -2.523E  DT2  25.5  NAEI  17.2 t CH4  improved grass 92 ha


cell near Nantwich: {'dairy': 6, 'beef_other': 10, 'sheep': 13, 'goats': 0} 1.4 t CH4


{'improved_ha': 908, 'semi_natural_ha': 2529, 'rough_grazing_ha': 23458, 'land_ha': 31614}


## Validation against the NAEI map and the shipped calibration

In [9]:
cal = await ask("calibration_report")
print("chosen allocation:", cal["chosen"])
for name, s in [("DT2 (hold-out)", cal["holdout"])] + list(cal["reference_holdout"].items()):
    print(f"{name:42s} r 1 km {s['r_1km']:.3f}  5 km {s['r_5km']:.3f}  10 km {s['r_10km']:.3f}  20 km {s['r_20km']:.3f}")
v = await ask("validate_naei", area={"bbox_bng": [300000, 300000, 400000, 400000]})
print("NW England 100 km box:", {k: round(x, 3) for k, x in v["skill"].items()})

chosen allocation: {'alpha_other': 0.0, 'block_km': 10.0, 'acn_weight': 0.5}
DT2 (hold-out)                             r 1 km 0.653  5 km 0.868  10 km 0.934  20 km 0.974
malek_all_layers_no_living_england         r 1 km 0.503  5 km 0.742  10 km 0.830  20 km 0.887
malek_grazing_layers_no_living_england     r 1 km 0.558  5 km 0.857  10 km 0.934  20 km 0.974


NW England 100 km box: {'r_1km': 0.613, 'r_5km': 0.836, 'r_10km': 0.924, 'r_20km': 0.962, 'dt2_t_ch4': 46894.54, 'naei_t_ch4': 57376.712, 'ratio_dt2_to_naei': 0.817}


## Learning a site from methane observations

In [10]:
async with Client(params) as client:      # one session, so the learned factors persist
    l = (await client.call_tool("assimilate_naei", {"area": crewe, "site": "crewe"})).structured_content
    print("prior", l["prior"], "\nposterior", {k: round(v, 3) for k, v in l["posterior"].items()})
    a = (await client.call_tool("area_methane", {"area": crewe, "site": "crewe"})).structured_content
    print(f"Crewe after learning: {a['total_t_ch4']:.0f} t CH4 ({a['provenance']['site_factors']})")
    t = (await client.call_tool("assimilate_total", {"area": lakes, "site": "lakes",
         "observed_t_ch4": 1000, "sd_t_ch4": 100})).structured_content
    print("Lake District from an observed total of 1000 +/- 100 t:", {k: round(v, 3) for k, v in t["posterior"].items()})
    print((await client.call_tool("reset_parameters", {"site": "crewe"})).structured_content["source"])

prior {'cattle': 1.0, 'sheep': 1.0, 'cattle_sd': 0.3, 'sheep_sd': 0.3} 
posterior {'cattle': 1.222, 'sheep': 1.011, 'cattle_sd': 0.095, 'sheep_sd': 0.3, 'correlation': -0.07}
Crewe after learning: 4301 t CH4 (learned 2026-11-15T19:12:12+00:00 (NAEI 2020 cells, 10 km around 53.1000N -2.4400E))
Lake District from an observed total of 1000 +/- 100 t: {'cattle': 1.263, 'sheep': 1.32, 'cattle_sd': 0.255, 'sheep_sd': 0.23, 'correlation': -0.52}
June Survey 2020 totals (no site learning)
